In [1]:
from neo.io import NixIO
import quantities as pq
filename = "results/network_data27_02_2025.nix"
with NixIO(filename, mode="rw") as io:
    block = io.read_block()
print(block.segments[0].spiketrains[0].times)
print(f"Number of segments: {len(block.segments)}")
print(f"Number of spike_trains: {len(block.segments[0].spiketrains)}")

# AT SOME POINT LETS DO A PREBINNING TO SAVE COMPUTE - BINNING EACH TIME TAKES TIME BUT NOT THAT MUCH


[48.0569 48.0693 48.1408 48.1428 48.3826 48.5236 48.7026 48.7375 48.7676
 48.7997 48.8017 48.9313 49.1222 49.1332 49.2584 49.2604 49.2871 49.3134
 49.4345 49.4577 49.4597 49.4617 49.6351 49.6916 49.6936 49.807  49.809
 49.811  49.9334 49.9354 49.9826 49.9909 50.1134 50.171  50.2017 50.2339
 50.2814 50.3313 50.3716 50.3736 50.4698 50.5705 50.5878 50.6787 50.82
 50.822  50.8384 50.9052 50.9078 50.9184 51.1047 51.1067 51.1387 51.2726
 51.2746 51.3265 51.3445 51.3465 51.3991 51.4046 51.5377 51.5397 51.5822
 51.5844 51.6773 51.7358 51.8124 51.8148 51.9381 51.9401 52.0494 52.0514
 52.0775 52.1872 52.1892 52.3982 52.4291 52.4311 52.4338 52.4358 52.4686
 52.5129 52.5772 52.5792 52.5821 52.5841 52.5861 52.6189 52.7775 52.872
 52.882  52.8996 52.9303 52.9338 52.9358 53.0852 53.2109 53.2183 53.2203
 53.2223 53.2306 53.3321 53.3413 53.3466 53.396  53.3995 53.4075 53.4095
 53.5365 53.5385 53.5897 53.7789 53.8677 53.8698 53.9841 53.9861 54.0011
 54.0031 54.1693 54.1713 54.3545 54.3888 54.4193 54.443

In [2]:
print(block.segments[0].spiketrains[0].annotations.keys())

dict_keys(['nix_name', 'neuron_id', 'forward_connections', 'weights', 'delays'])


In [ ]:
# metadata = {}
# for i, values in enumerate(block.segments[0].spiketrains):
#     metadata[i] = values.annotations

# triples = []
# sparse_triples = []
# # Get triples
# alpha = 5 * pq.s
# for initial_neuron in range(len(metadata)):
#     second_neurons = metadata[initial_neuron]['forward_connections']
#     for i2, second_neuron in enumerate(second_neurons):
#         t1_2 = metadata[initial_neuron]['delays'][i2]
#         third_neurons = metadata[second_neuron]['forward_connections']
#         for i3, third_neuron in enumerate(third_neurons):
#             t2_3 = metadata[second_neuron]['delays'][i3]
#             if third_neuron in metadata[initial_neuron]['forward_connections']:
#                 location_of_third = metadata[initial_neuron]['forward_connections'].index(third_neuron)
#                 t1_3 = metadata[initial_neuron]['delays'][location_of_third]
#                 triples.append((initial_neuron, second_neuron, third_neuron))
#                 if t1_2 + t2_3 - t1_3 < alpha:
#                     sparse_triples.append((initial_neuron, second_neuron, third_neuron))

In [ ]:
import tqdm
import time
spiketrains = block.segments[0].spiketrains
metadata = {}
forward_connections = {}  # (source, target) -> delay
forward_sets = {}  # neuron -> set of targets
forward_weights = {} # (source, target) -> weight
side_connections = {} # (source, target) -> delay
side_sets = {} # neuron -> set of targets
side_weights = {} # (source, target) -> weight

n_neurons = 64**2
print(n_neurons)
# Build efficient lookup structures
#build two lookup structures one for the 2->3 and another for the 1->2. 
print("building filtered lookup structures")
for i, values in tqdm.tqdm(enumerate(spiketrains),desc="building filtered lookup structures"):
    metadata[i] = values.annotations
    connections = values.annotations['forward_connections']
    delays = values.annotations['delays']
    connection_weights = values.annotations['weights']
    this_layer_start = (i//n_neurons) * n_neurons
    next_layer_start = this_layer_start + n_neurons
    # Initialize a filtered set for this neuron
    forward_sets[i] = set()
    side_sets[i] = set()
    # Filter connections with weight threshold
    for j, (target, weight) in enumerate(zip(connections, connection_weights)):
        if weight > 0.5:
            if target>=(next_layer_start):  # Only keep connections with weights >= 0.5 and in the next layer
                forward_sets[i].add(target)
                forward_connections[(i, target)] = delays[j]
                forward_weights[(i, target)] = weight
            elif target>=this_layer_start: # Only keep connections with weights >= 0.5 and in the same layer
                side_sets[i].add(target)
                side_connections[(i, target)] = delays[j]
                side_weights[(i, target)] = weight
print("done")
print(f"forward len {len(forward_connections)}")
print(f"side len {len(side_connections)}")

triples = []
sparse_triples = []
alpha = 3 * pq.s

# Find triples efficiently with set operations
print("Finding triples with weight-filtered connections...")
for neuron1 in tqdm.tqdm(metadata.keys(), desc="Processing neurons"):
    # Get filtered targets for neuron1
    targets_of_neuron1 = forward_sets[neuron1]
    
    for neuron2 in targets_of_neuron1:
        t1_2 = forward_connections[(neuron1, neuron2)]
        
        # Find all neurons that both neuron1 and neuron2 connect to (with weight ≥ 0.5)
        shared_targets = targets_of_neuron1.intersection(side_sets[neuron2])
        
        # Process only the valid shared targets
        for neuron3 in shared_targets:
            t1_3 = forward_connections[(neuron1, neuron3)]
            t2_3 = side_connections[(neuron2, neuron3)]
            
            triples.append((neuron1, neuron2, neuron3))
            if abs(t1_2 + t2_3 - t1_3) < alpha:
                sparse_triples.append((neuron1, neuron2, neuron3))

print(f"Found {len(triples)} triples, {len(sparse_triples)} of which are sparse triples")




In [2]:


import quantities as pq
import numpy as np
import tqdm  # Use regular tqdm which works in any environment

def add_silent_gaps(spiketrains, segment_duration=250*pq.ms, gap_duration=50*pq.ms):
    """
    Add silent gaps between segments of spike trains.
    
    Parameters:
    -----------
    spiketrains : list of neo.SpikeTrain
        The spike trains to modify
    segment_duration : Quantity
        The duration of each segment (default: 250ms)
    gap_duration : Quantity
        The duration of each gap (default: 50ms)
    
    Returns:
    --------
    list of neo.SpikeTrain
        Modified spike trains with gaps inserted
    """
    modified_spiketrains = []
    total_spikes = sum(len(st) for st in spiketrains)
    
    print(f"Processing {len(spiketrains)} spike trains with {total_spikes} total spikes")
    print(f"Adding {gap_duration} gaps every {segment_duration}")
    
    for st in tqdm.tqdm(spiketrains, desc="Processing spike trains"):
        # Ensure we're working with milliseconds
        times_ms = st.rescale('ms').magnitude
        segment_duration_ms = segment_duration.rescale('ms').magnitude
        gap_duration_ms = gap_duration.rescale('ms').magnitude
        
        # Calculate which segment each spike belongs to
        segments = np.floor(times_ms / segment_duration_ms).astype(int)
        
        # Calculate the offset for each spike based on its segment
        offsets = segments * gap_duration_ms
        
        # Apply the offset to get new times
        new_times_ms = times_ms + offsets
        
        # Calculate new t_stop
        total_segments = int(np.ceil(st.t_stop.rescale('ms').magnitude / segment_duration_ms))
        new_t_stop_ms = st.t_stop.rescale('ms').magnitude + (total_segments * gap_duration_ms)
        
        # Create new spike train with original units
        original_units = st.units
        new_st = st.duplicate_with_new_data(
            (new_times_ms * pq.ms).rescale(original_units), 
            t_stop=(new_t_stop_ms * pq.ms).rescale(original_units)
        )
        
        # Copy over annotations and other attributes
        new_st.annotations.update(st.annotations)
        modified_spiketrains.append(new_st)
    
    # Summary of changes
    original_duration = max(st.t_stop for st in spiketrains)
    modified_duration = max(st.t_stop for st in modified_spiketrains)
    print(f"Processing complete: Duration extended from {original_duration} to {modified_duration}")
    
    return modified_spiketrains

modified_spikes = add_silent_gaps(block.segments[0].spiketrains)

Processing 16384 spike trains with 6757943 total spikes
Adding 50.0 ms gaps every 250.0 ms


Processing spike trains: 100%|██████████| 16384/16384 [06:40<00:00, 40.90it/s]


Processing complete: Duration extended from 68.0 s to 81.60000000000001 s


In [3]:
import pickle

# # Save sparse_triples to a pickle file
# with open('sparse_triples.pkl', 'wb') as f:
#     pickle.dump(sparse_triples, f)

# print("sparse_triples have been pickled and saved to 'sparse_triples.pkl'")

# retrieve the sparse_triples on later run
with open('sparse_triples.pkl', 'rb') as f:
    sparse_triples = pickle.load(f)

In [4]:
test_data = []

def shift_spiketrain(spiketrain, shift):
    new_times = spiketrain.times - shift
    new_times = new_times[new_times >= spiketrain.t_start]
    shifted_st = spiketrain.duplicate_with_new_data(new_times,t_start=spiketrain.t_start, t_stop=spiketrain.t_stop)
    return shifted_st
for i, triple in enumerate(sparse_triples):
    if i%500==0:
        print(f"Processing triple {i}/{len(sparse_triples)}")
    index_1_2 = metadata[triple[0]]['forward_connections'].index(triple[1])
    index_1_3 = metadata[triple[0]]['forward_connections'].index(triple[2])
    index_2_3 = metadata[triple[1]]['forward_connections'].index(triple[2])
    t1_2 = metadata[triple[0]]['delays'][index_1_2] /1000
    t1_3 = metadata[triple[0]]['delays'][index_1_3] /1000
    t2_3 = metadata[triple[1]]['delays'][index_2_3] /1000
    st1 = modified_spikes[triple[0]]
    if len(st1)==0:
        continue
    st2 = modified_spikes[triple[1]]
    if len(st2)==0:
        continue
    st3 = modified_spikes[triple[2]]
    if len(st3)==0:
        continue
    shifted_1 = st1
    shifted_2 = shift_spiketrain(st2, t1_2)
    shifted_3 = shift_spiketrain(st3, (t1_3 + max(t1_2+t2_3-t1_3,0*pq.s)))

    test_data.append({'data':[shifted_1,shifted_2, shifted_3]})

print(f"Processed {len(test_data)} triples")

Processing triple 0/78445


NameError: name 'metadata' is not defined

In [70]:
import ray
import time
import numpy as np
import elephant
import quantities as pq
from datetime import datetime
import ctypes
import logging

# Configure logging
logging.basicConfig(level=logging.DEBUG)

ES_CONTINUOUS      = 0x80000000  # Informs the system that the state being set should remain in effect until the next call.
ES_SYSTEM_REQUIRED = 0x00000001  # Forces the system to be in the working state by resetting the system idle timer.
ES_DISPLAY_REQUIRED= 0x00000002  # Forces the display to be on by resetting the display idle timer.

# Prevent sleep: This call tells Windows to keep the system and display awake.
ctypes.windll.kernel32.SetThreadExecutionState(
    ES_CONTINUOUS | ES_SYSTEM_REQUIRED | ES_DISPLAY_REQUIRED
)

ray.shutdown()
# Fixed: removed ray.logging reference
ray.init(num_cpus=3, logging_level=logging.DEBUG)

@ray.remote
class ProgressActor:
    def __init__(self, total_tasks):
        self.total = total_tasks
        self.completed = 0
        self.start_time = time.time()

    def update(self):
        self.completed += 1
        if self.completed % 5 == 0 or self.completed == self.total:
            elapsed = time.time() - self.start_time
            tasks_per_sec = self.completed / elapsed
            eta = (self.total - self.completed) / tasks_per_sec if tasks_per_sec > 0 else "unknown"
            eta_str = str(eta) if isinstance(eta, str) else f"{eta:.2f} sec"
            print(f"[{datetime.now().strftime('%H:%M:%S')}] "
                  f"Completed {self.completed}/{self.total} ({self.completed/self.total*100:.1f}%) "
                  f"- Rate: {tasks_per_sec:.2f} tasks/sec - ETA: {eta_str}")
        return self.completed

    def get_completed(self):
        return self.completed

@ray.remote
def process_triple(data_idx, triple_data, progress_actor):
    """
    Process a single triple of spike trains
    
    Args:
        process_id: Identifier for this process
        triple_data: Dictionary containing spike train data 
        progress_actor: Actor to track progress
    """
    try:
        # Extract spike triple data
        spike_triple = triple_data['data']        
        # Calculate firing rates and CV values
        firing_rates = []
        for i, st in enumerate(spike_triple):
            rate = elephant.statistics.mean_firing_rate(st)
            if hasattr(rate, 'item'):
                firing_rates.append(rate.item())
            else:
                firing_rates.append(float(rate))
        
        cv_values = [elephant.statistics.cv(st) for st in spike_triple]
        
        # Create features dictionary
        features = {
            'index': data_idx,
            'min_rate': min(firing_rates),
            'max_rate': max(firing_rates),
            'mean_rate': np.mean(firing_rates),
            'min_cv': min(cv_values),
            'avg_cv': np.mean(cv_values),
            'cv_std': np.std(cv_values)
        }
        
        # Calculate p-value spectrum
        outcome = elephant.spade.pvalue_spectrum(
            spiketrains=spike_triple,
            bin_size=1*pq.ms,
            winlen=3,
            min_spikes=3,
            max_spikes=3,
            min_neu=3,
            n_surr=200,
            dither=5*pq.ms
        )
        ray.get(progress_actor.update.remote())
        return data_idx, features, outcome
    
    except Exception as e:
        print(f"ERROR [{data_idx}]: {type(e).__name__}: {str(e)}")
        import traceback
        print(f"ERROR [{data_idx}]: {traceback.format_exc()}")
        return data_idx, None, None

def make_sample_spectra(spike_data_list, sample_size=100):
    """
    Process a sample of spike data triplets
    
    Args:
        spike_data_list: List of spike data dictionaries 
        sample_size: Number of samples to process
    """
    print(f"DEBUG: spike_data_list type: {type(spike_data_list)}, length: {len(spike_data_list)}")
    sample_idx = range(len(spike_data_list))
    
    # Select indices for processing
    if len(spike_data_list) > sample_size:
        selected_indices = np.random.choice(len(spike_data_list), size=sample_size, replace=False)
        print(f"DEBUG: Selected {sample_size} random indices from {len(spike_data_list)} total")
    else:
        selected_indices = range(len(spike_data_list))
        print(f"DEBUG: Using all {len(spike_data_list)} indices")
    
    # Initialize progress tracking
    progress_actor = ProgressActor.remote(len(selected_indices))
    print(f"Starting processing of {len(selected_indices)} samples")
    
    # Submit tasks to Ray
    start = time.time()
    futures = []
    
    for data_idx in selected_indices:
        # Debug the data before submitting
        data_to_process = spike_data_list[data_idx]        
        # Submit the Ray task
        futures.append(process_triple.remote(data_idx, data_to_process, progress_actor))
    
    # Get results
    results = ray.get(futures)
    data_idxs, features_list, spectra_list = zip(*results)
    
    # Report completion
    end = time.time()
    print(f"Processing completed in {end-start:.2f} seconds")
    return process_ids, features_list, spectra_list

print("Starting sample spectra calculation")
process_ids, features, spectra = make_sample_spectra(test_data, sample_size=40)
print("Sample spectra calculation complete")
print(ray.timeline())
ray.shutdown()

2025-03-06 15:46:32,869	DEBUG worker.py:1592 -- Could not import resource module (on Windows)


2025-03-06 15:46:36,825	DEBUG node.py:293 -- Setting node ID to 4abb06ba5b0d4c3d56d55bb4483a8a5b463c7dd2fefa7db5ce70c7d5
2025-03-06 15:46:37,067	DEBUG node.py:1401 -- Process STDOUT and STDERR is being redirected to C:\Users\reidj\AppData\Local\Temp\ray\session_2025-03-06_15-46-36_797451_43252\logs.
2025-03-06 15:46:45,526	DEBUG node.py:1430 -- Process STDOUT and STDERR is being redirected to C:\Users\reidj\AppData\Local\Temp\ray\session_2025-03-06_15-46-36_797451_43252\logs.
2025-03-06 15:46:45,759	DEBUG npu.py:60 -- Could not import AscendCL: No module named 'acl'
2025-03-06 15:46:45,824	DEBUG services.py:2140 -- Determine to start the Plasma object store with 0.59 GB memory using C:\Users\reidj\AppData\Local\Temp.
2025-03-06 15:46:46,124	INFO worker.py:1841 -- Started a local Ray instance.


Starting sample spectra calculation
DEBUG: spike_data_list type: <class 'list'>, length: 46880
DEBUG: Selected 40 random indices from 46880 total
Starting processing of 40 samples
(process_triple pid=43208) DEBUG [13557]: Outcome type: <class 'list'>
(process_triple pid=42560) DEBUG [28239]: Outcome type: <class 'list'>
(process_triple pid=23836) DEBUG [37811]: Outcome type: <class 'list'>
(process_triple pid=43208) DEBUG [45326]: Outcome type: <class 'list'>
(ProgressActor pid=9468) [15:47:53] Completed 5/40 (12.5%) - Rate: 0.09 tasks/sec - ETA: 411.46 sec
(process_triple pid=23836) DEBUG [37908]: Outcome type: <class 'list'>
(process_triple pid=42560) DEBUG [2872]: Outcome type: <class 'list'>
(process_triple pid=23836) DEBUG [24013]: Outcome type: <class 'list'>
(process_triple pid=42560) DEBUG [3789]: Outcome type: <class 'list'>
(process_triple pid=43208) DEBUG [18524]: Outcome type: <class 'list'>
(ProgressActor pid=9468) [15:48:48] Completed 10/40 (25.0%) - Rate: 0.09 tasks/sec 

In [5]:
import ray
import time
import numpy as np
import elephant
import quantities as pq
from datetime import datetime
import logging

ray.shutdown()
# Fixed: removed ray.logging reference
ray.init(num_cpus=3, logging_level=logging.DEBUG)

@ray.remote
class ProgressActor:
    def __init__(self, total_tasks):
        self.total = total_tasks
        self.completed = 0
        self.start_time = time.time()

    def update(self):
        self.completed += 1
        if self.completed % 1 == 0 or self.completed == self.total:
            elapsed = time.time() - self.start_time
            tasks_per_sec = self.completed / elapsed
            eta = (self.total - self.completed) / tasks_per_sec if tasks_per_sec > 0 else "unknown"
            eta_str = str(eta) if isinstance(eta, str) else f"{eta:.2f} sec"
            print(f"[{datetime.now().strftime('%H:%M:%S')}] "
                  f"Completed {self.completed}/{self.total} ({self.completed/self.total*100:.1f}%) "
                  f"- Rate: {tasks_per_sec:.2f} tasks/sec - ETA: {eta_str}")
        return self.completed

    def get_completed(self):
        return self.completed

@ray.remote
def process_triple(data_idx, triple_data, progress_actor):
    """
    Process a single triple of spike trains
    
    Args:
        data_idx: Identifier for this process
        triple_data: Dictionary containing spike train data 
        progress_actor: Actor to track progress
    """
    try:
        # Extract spike triple data
        spike_triple = triple_data['data']        
        # Calculate p-value spectrum
        outcome = elephant.spade.spade(
            spiketrains=spike_triple,
            bin_size=1*pq.ms,
            winlen=3,
            min_spikes=3,
            max_spikes=3,
            min_neu=3,
            n_surr=200,
            dither=5*pq.ms
        )
        ray.get(progress_actor.update.remote())
        return data_idx, outcome
    
    except Exception as e:
        print(f"ERROR [{data_idx}]: {type(e).__name__}: {str(e)}")
        import traceback
        print(f"ERROR [{data_idx}]: {traceback.format_exc()}")
        return data_idx, None
def get_results(spike_data_list):
    """
    Process a sample of spike data triplets
    
    Args:
        spike_data_list: List of spike data dictionaries 
        sample_size: Number of samples to process
    """
    # Initialize progress tracking
    progress_actor = ProgressActor.remote(len(spike_data_list))
    print(f"Starting processing of {len(spike_data_list)} samples")
    
    # Submit tasks to Ray
    start = time.time()
    futures = []
    
    for data_idx in range(len(spike_data_list)):
        # Debug the data before submitting
        data_to_process = spike_data_list[data_idx]        
        # Submit the Ray task
        futures.append(process_triple.remote(data_idx, data_to_process, progress_actor))
    
    # Get results
    results = ray.get(futures)
    
    # Report completion
    end = time.time()
    print(f"Processing completed in {end-start:.2f} seconds")
    return results

results = get_results(test_data)
print("all done")
print(ray.timeline())
ray.shutdown()

c:\Users\reidj\Dropbox\dphil\programming\spikes\.datanalysis\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-03-07 17:30:58,597	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2025-03-07 17:30:59,297	DEBUG worker.py:1592 -- Could not import resource module (on Windows)
2025-03-07 17:31:02,673	DEBUG node.py:293 -- Setting node ID to 2057ab468e9c9329e1b822af0002c83b5dd723ea204a0922be85c5f2
2025-03-07 17:31:02,727	DEBUG node.py:1401 -- Process STDOUT and STDERR is being redirected to C:\Users\reidj\AppData\Local\Temp\ray\session_2025-03-07_17-31-02_648968_3312\logs.
2025-03-07 17:31:09,363	DEBUG node.py:1430 -- Process STDOUT and STDERR is being redirected to C:\Users\reidj\AppData\Local\Temp\ray\session_2025-03-07_

NameError: name 'test_data' is not defined